# Episodic Memory with Write-Time Reflection

This notebook demonstrates `ReflectiveMemory` and `JSONMemoryStore` working together
to give an `LLMAgent` episodic memory that distils a short lesson from each completed
task at write time.

The example is drawn directly from the
[Reflexion paper (Shinn et al., 2023)](https://arxiv.org/abs/2303.11366), which
evaluates the pattern on coding tasks: the agent writes code, runs it against
expected output, and reflects on what went wrong. Recalled reflections from past
mistakes guide the agent on structurally similar tasks later.

Two properties are illustrated:

- **Write-time reflection.** After each task the agent records an episode.
  Before the episode is stored, `ReflectiveMemory` calls the LLM to produce a
  one-sentence lesson from the task and its outcome. That lesson is attached to the
  episode under `additional_data["reflection"]` and rendered automatically inside a
  `<reflection>` tag whenever the episode is displayed.
- **Recalled lessons.** When a new task arrives, `recall()` surfaces the three most
  recent episodes. Because each episode now carries a pre-distilled lesson, the agent
  can recognise a familiar bug pattern and write correct code on the first attempt
  rather than discovering the mistake at run time.

The tasks in this notebook are based on three classic Python gotchas — mutable
default arguments, late-binding closures, and floating-point equality — each of which
reliably produces a wrong answer on the first attempt.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code in this notebook you need Ollama installed locally with its
service running. Download Ollama from https://ollama.com/download, then start the
service by running `ollama serve` in a terminal.

## Setup

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

In [ ]:
import logging
from pathlib import Path

from tqdm.asyncio import tqdm

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory import JSONMemoryStore, ReflectiveMemory
from llm_agents_from_scratch.tools import PythonInterpreterTool

## Creating the Agent with Memory

`PythonInterpreterTool` lets the agent write and run inline Python code directly.
The agent uses it to test its solutions and observe whether the output matches what
is expected. `JSONMemoryStore` persists the resulting episodes to disk.
`ReflectiveMemory` wraps the store: before each episode is saved it calls
`llm.complete()` to distil a one-sentence lesson from the task and result, then
stores that lesson under `episode.additional_data["reflection"]`.

In [ ]:
STORE_DIR = Path(".")
(STORE_DIR / "reflective_episodes.jsonl").unlink(missing_ok=True)

store = JSONMemoryStore(dir=STORE_DIR, filename="reflective_episodes.jsonl")
llm = OllamaLLM(model="qwen3:14b", think=False)
memory = ReflectiveMemory(store=store, llm=llm, n=3)
agent = LLMAgent(
    llm=llm,
    tools=[PythonInterpreterTool()],
    memories=[memory],
)

## Part 1 — Learning from Classic Python Bugs

Three tasks, each built around a Python gotcha that reliably produces a wrong result
on the first attempt:

1. **Mutable default argument** — using `[]` as a default parameter creates a shared
   list that accumulates across calls.
2. **Late-binding closure** — lambdas in a list comprehension capture the loop
   variable by reference, so all functions return the last value of the variable.
3. **Floating-point equality** — `0.1 + 0.2 == 0.3` is `False` in Python due to
   IEEE 754 representation.

The first task runs with full logging so you can see the agent write buggy code, run
it, observe the wrong output, debug, and fix. The remaining two run concurrently with
logging silenced. After all three complete, each episode is stored with a reflection.

In [ ]:
enable_console_logging(logging.INFO)

task1 = Task(
    instruction=(
        "Implement a Python function with signature "
        "`def add_to_list(item, lst=[])` that appends item to lst "
        "and returns lst. "
        "Then test it by calling add_to_list(1), add_to_list(2), "
        "add_to_list(3) in the same script. "
        "Each call should print a list containing only the item "
        "passed in that call: [1], [2], [3]."
    ),
)
result1 = await agent.run(task1)
print(result1)

In [ ]:
logging.disable(logging.INFO)

task2 = Task(
    instruction=(
        "Create a list called `funcs` containing 5 lambda functions using a "
        "list comprehension over `range(5)`. The i-th function should return "
        "i * i when called. Verify by printing funcs[0](), funcs[2](), and "
        "funcs[4]() — expected outputs are 0, 4, and 16."
    ),
)
task3 = Task(
    instruction=(
        "Write Python code that checks whether 0.1 + 0.2 is equal to 0.3. "
        "Print the raw result of 0.1 + 0.2, then print whether it equals 0.3 "
        "using ==, then fix the comparison so it prints True."
    ),
)

await tqdm.gather(
    agent.run(task2),
    agent.run(task3),
    desc="Running coding tasks",
)

In [ ]:
logging.disable(logging.NOTSET)

print(await memory.summary())

## The RecencyMemory Baseline

Before looking at the reflections, consider what `RecencyMemory` would surface
for a new task. A `RecencyMemory` stores no additional data. The recalled episodes
contain the full task narrative — the agent's reasoning, the buggy code, the wrong
output, the fix — but the lesson is buried inside that transcript. A future agent
would have to re-read the entire debugging session to extract what went wrong.

In [ ]:
print("What RecencyMemory would recall (same episodes, no reflections):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    ep_without_reflection = Episode(
        task=ep.task,
        rollout=ep.rollout,
        result=ep.result,
        completed_at=ep.completed_at,
    )
    print(str(ep_without_reflection))
    print()

## Part 2 — Inspecting the Reflections

Now look at the same three episodes as stored by `ReflectiveMemory`. Each carries
a `<reflection>` tag with the one-sentence lesson distilled at write time. The bug,
the root cause, and the fix are captured in a single line rather than buried in a
multi-step debugging transcript.

In [ ]:
print("What ReflectiveMemory recalls (same episodes, with reflections):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    print(str(ep))
    print()

## Part 3 — A Recalled Lesson Prevents a Bug

The final task introduces the same mutable default argument pattern from Part 1,
but phrased as a fresh problem. Without prior context the agent would implement the
given signature as-is, discover the bug at run time, and then fix it.

With the three Part 1 reflections in its system prompt, the agent has already seen
the lesson: mutable defaults create shared state across calls, and `None` with an
in-function initialisation is the correct fix. Check the logs: an agent that applies
the recalled lesson writes correct code on the first tool call; an agent without the
context takes at least two.

In [ ]:
task4 = Task(
    instruction=(
        "A colleague started writing a stack helper with signature "
        "`def push(item, stack=[])`. Complete and test it so that calling "
        "push('a'), push('b'), push('c') prints ['a'], ['b'], ['c'] "
        "respectively — each call must start from an empty stack."
    ),
)

print("Episodes that will be recalled for this task:\n")
recalled = await memory.store.read_recent(memory.n)
for ep in recalled:
    print(str(ep))
    print()

In [ ]:
enable_console_logging(logging.INFO)

result4 = await agent.run(task4)
print(result4)

## Key Takeaway

`ReflectiveMemory` converts task experience into transferable lessons. The Reflexion
paper frames this as verbal reinforcement learning: rather than updating model
weights, the agent stores natural-language feedback from each trial and retrieves it
when facing structurally similar problems.

The comparison in Parts 1 and 2 makes the difference concrete. Raw `RecencyMemory`
episodes surface the full debugging transcript — buggy code, error output, revised
code, final output — but the lesson is implicit. `ReflectiveMemory` surfaces it in
a single sentence alongside the episode, so a future task can act on it without
replaying the entire debugging session.

The separation of concerns is the same as in `RecencyMemory` and `SimilarityMemory`:
`ReflectiveMemory` decides *what* to generate, *what* to attach, and *how* to format
the recalled context; `JSONMemoryStore` decides *how* to persist and retrieve
episodes. Swapping the store does not affect the reflection logic, and any
`BaseMemoryStore` that implements `read_recent()` works as a drop-in backend.